# Comparative Sentiment Analysis: Indonesian E-Commerce Reviews (Scopus Pipeline)
## Notebook 04: IndoBERT Fine-Tuning (Colab & Local GPU Accelerated)

```text
Author: Muhammad Farrel Haidar
Project: AI PR & KOL Specialist DSS
Date: 2026-10-14
Study: Comparative Sentiment Analysis (IndoBERT vs Tree-Based Models)
Model: indobenchmark/indobert-base-p1 (Transformer Architecture, 110M Parameters)
Hardware: Local NVIDIA RTX 4070 (8GB VRAM) / Google Colab (Tesla T4 / A100)
```

---
### 1. Landasan Teoretis & Formulasi Matematis Transformer

#### a. Mekanisme Multi-Head Scaled Dot-Product Self-Attention
Inti arsitektur IndoBERT adalah mekanisme atensi kontekstual bidirectional yang memetakan keterkaitan semantik antar-kata secara dinamis:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}}\right) V$$

di mana $Q$ (Query), $K$ (Key), dan $V$ (Value) adalah proyeksi linear representasi teks berdimensi $d_k$.

#### b. Fungsi Objektif: Categorical Cross-Entropy Loss
Optimisasi bobot selama fine-tuning 3-kelas menggunakan fungsi *cross-entropy loss*:

$$\mathcal{L}_{\text{CE}} = -\sum_{i=1}^{N} \sum_{c=0}^{2} y_{i,c} \log(\hat{y}_{i,c})$$

di mana $y_{i,c} \in \{0, 1\}$ adalah indikator one-hot kelas sebenarnya (Negatif, Netral, Positif) dan $\hat{y}_{i,c}$ adalah probabilitas prediksi dari fungsi softmax pada pooling token `[CLS]`.

In [ ]:
# Author: Muhammad Farrel Haidar
# Project: AI PR & KOL Specialist DSS
# Date: 2026-10-14

# 1. Environment & GPU Hardware Detection (Colab / Local Auto-Detect)
import os
import sys
import time
import torch

# Google Colab Mount Setup (Jika dijalankan di Colab)
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Mounted Google Drive successfully.")
    !pip install -q transformers==4.44.2 datasets==2.21.0 evaluate accelerate

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Active Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM (GB)  : {round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)} GB")

### 2. Import Libraries & Dataset Tokenization

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    set_seed
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score

set_seed(42)

# Load Split Data
data_dir = "data/splits"
train_df = pd.read_csv(f"{data_dir}/train.csv")
val_df = pd.read_csv(f"{data_dir}/val.csv")
test_df = pd.read_csv(f"{data_dir}/test.csv")

print(f"Train: {len(train_df):,}, Val: {len(val_df):,}, Test: {len(test_df):,}")

MODEL_NAME = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, item):
        text = str(self.texts[item])
        label = int(self.labels[item])
        encoding = self.tokenizer(text, truncation=True, padding="max_length", max_length=self.max_len, return_tensors="pt")
        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(label, dtype=torch.long)
        }

train_ds = ReviewDataset(train_df["cleaned_text"].fillna(""), train_df["label"].values, tokenizer)
val_ds = ReviewDataset(val_df["cleaned_text"].fillna(""), val_df["label"].values, tokenizer)
test_ds = ReviewDataset(test_df["cleaned_text"].fillna(""), test_df["label"].values, tokenizer)
print("Datasets ready for training.")

### 3. Hyperparameter Configuration & Custom Compute Metrics

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(labels, preds, average="weighted", zero_division=0)
    return {
        "accuracy": acc,
        "precision_macro": p_macro,
        "recall_macro": r_macro,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted
    }

output_dir = "/content/drive/MyDrive/indobert_ecommerce_sentiment" if IN_COLAB else "models/indobert_finetuned"
training_args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    seed=42,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)
print("Trainer initialized.")

### 4. Pelatihan Model & Evaluasi Test Set

In [ ]:
# Jalankan pelatihan
t0 = time.time()
trainer.train()
print(f"Training finished in {time.time()-t0:.2f}s")

# Simpan model final
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

# Prediksi Test Set
preds = trainer.predict(test_ds)
y_pred = np.argmax(preds.predictions, axis=-1)
y_true = test_df["label"].values

print(f"Test Accuracy: {accuracy_score(y_true, y_pred):.4f}")
p_m, r_m, f1_m, _ = precision_recall_fscore_support(y_true, y_pred, average="macro")
print(f"Test Macro-F1: {f1_m:.4f}")

### 5. Visualisasi Confusion Matrix & Kurva Konvergensi

In [ ]:
# Visualisasi Confusion Matrix IndoBERT
cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Negatif", "Netral", "Positif"])
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix: IndoBERT Fine-Tuned (Test Set)")
plt.show()